In [1]:
import sys
from pathlib import Path
import torch

ROOT = Path(r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification")

sys.path.insert(0, str(ROOT))

In [2]:
from src.models.cbam_cnn import (
    CBAMCNN
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [3]:
model = CBAMCNN()

model.load_state_dict(
    torch.load(
        "../models/federated_cbam_best.pth",
        map_location=device
    )
)

model.to(device)

model.eval()

C:\Users\gagan\AppData\Local\Temp\ipykernel_27384\2965872004.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


CBAMCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilatio

In [4]:
all_labels = []
all_predictions = []
all_probabilities = []

In [5]:
from src.preprocessing.dataset_loader import build_dataset_index
from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)
from src.preprocessing.splitter import create_stratified_split

In [6]:
from pathlib import Path

ROOT = Path(
    r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification"
)

train_dir = ROOT / "data" / "raw" / "train"
test_dir = ROOT / "data" / "raw" / "test"

train_paths, train_labels = build_dataset_index(train_dir)
test_paths, test_labels = build_dataset_index(test_dir)

print(len(train_paths), len(train_labels))

5216 5216


In [7]:
print("Number of train images:", len(train_paths))
print("Number of train labels:", len(train_labels))

Number of train images: 5216
Number of train labels: 5216


In [8]:
(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

In [9]:
(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

In [10]:
(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [11]:
print(len(test_loader.dataset))

624


In [12]:
with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )[:,1]

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

9.899295037030242e-07 0.9999997615814209
0.4157944321632385 0.999855637550354
6.523008778458461e-05 0.999981164932251
0.4347016513347626 0.9998915195465088
0.00025322780129499733 0.9999196529388428
0.4282244145870209 0.9998038411140442
3.587391984183341e-05 0.9999854564666748
0.4333053529262543 0.9999680519104004
1.4673626935746142e-07 1.0
0.34881606698036194 0.9999908208847046
6.584799848496914e-05 0.9999819993972778
0.41543498635292053 0.9998292922973633
4.306338450987823e-05 0.999988317489624
0.4357317388057709 0.9998559951782227
9.535269782645628e-05 0.9999587535858154
0.4466402530670166 0.9997707009315491
2.0768684407812543e-05 0.9999920129776001
0.5302908420562744 0.9993941783905029
3.5883022064808756e-05 0.999985933303833
0.5272313356399536 0.9989972710609436
9.69897155300714e-06 0.999996542930603
0.5135740041732788 0.9994950294494629
6.013728580001043e-06 0.9999979734420776
0.5034670233726501 0.999946117401123
2.125532091667992e-06 0.9999992847442627
0.43341264128685 0.99992501

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = {

    "accuracy":
        accuracy_score(
            all_labels,
            all_predictions
        ),

    "precision":
        precision_score(
            all_labels,
            all_predictions
        ),

    "recall":
        recall_score(
            all_labels,
            all_predictions
        ),

    "f1_score":
        f1_score(
            all_labels,
            all_predictions
        ),

    "roc_auc":
        roc_auc_score(
            all_labels,
            all_probabilities
        )
}

In [14]:
print("\n=== FEDERATED TEST RESULTS ===\n")

for key, value in results.items():

    print(
        f"{key}: {value:.4f}"
    )


=== FEDERATED TEST RESULTS ===

accuracy: 0.7965
precision: 0.7646
recall: 0.9744
f1_score: 0.8568
roc_auc: 0.9161


In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

[[117 117]
 [ 10 380]]


In [16]:
from sklearn.metrics import classification_report

print(

    classification_report(
        all_labels,
        all_predictions
    )

)

              precision    recall  f1-score   support

           0       0.92      0.50      0.65       234
           1       0.76      0.97      0.86       390

    accuracy                           0.80       624
   macro avg       0.84      0.74      0.75       624
weighted avg       0.82      0.80      0.78       624



In [17]:
import json

ROOT = Path(
    r"C:\Users\gagan\Desktop\Federated Learning\federated-xray-classification"
)

with open(
    ROOT / "results" / "metrics" / "federated_results.json",
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )